In [2]:
!pip install mne AutoReject

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 74.3 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [1]:
import mne
import numpy as np
from autoreject import AutoReject

def preprocess_eeg(subjects, runs, annot_mapping, global_event_id):
    raws = []
    for subject in subjects:
        raw_fnames = mne.datasets.eegbci.load_data(subject, runs)
        for f in raw_fnames:
            raws.append(mne.io.read_raw_edf(f, preload=True))

    raw = mne.concatenate_raws(raws)

    # standaryzacja i car
    mne.datasets.eegbci.standardize(raw)
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage)
    raw.set_eeg_reference('average', projection=False)

    # Notch i Band-pass
    raw.notch_filter(freqs=[60.0])
    raw.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin')

    # usuwanie artefaktów ocznych (ICA)
    ica = mne.preprocessing.ICA(n_components=15, random_state=42, max_iter='auto')
    ica.fit(raw)
    eog_indices, _ = ica.find_bads_eog(raw, ch_name='Fp1')
    ica.exclude = eog_indices
    ica.apply(raw)

    # mapowanie unikalnych nazw klas
    raw.annotations.rename(annot_mapping)
    events, event_id = mne.events_from_annotations(raw, event_id=global_event_id)

    picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)
    epochs = mne.Epochs(raw, events, event_id, tmin=-0.5, tmax=4.0, proj=True,
                        picks=picks, baseline=(None, 0), preload=True)

    # AutoReject
    ar = AutoReject(random_state=42)
    epochs = ar.fit_transform(epochs, return_log=False)

    return epochs

In [2]:
subjects = [1, 2, 3, 4, 5]

runs_A = [4, 8, 12]  # Wyobraźnia: pięści
runs_B = [6, 10, 14] # Wyobraźnia: obie pięści / obie stopy

mapping_A = {'T0': 'Rest', 'T1': 'Left_Fist', 'T2': 'Right_Fist'}
mapping_B = {'T0': 'Rest', 'T1': 'Both_Fists', 'T2': 'Both_Feet'}

unified_event_id = {
    'Rest': 1,
    'Left_Fist': 2,
    'Right_Fist': 3,
    'Both_Fists': 4,
    'Both_Feet': 5
}

print("Przetwarzanie Datasetu A (sesje 4, 8, 12)...")
epochs_A = preprocess_eeg(subjects, runs_A, mapping_A, unified_event_id)

print("Przetwarzanie Datasetu B (sesje 6, 10, 14)...")
epochs_B = preprocess_eeg(subjects, runs_B, mapping_B, unified_event_id)

print("Łączenie w Dataset C (wszystkie sesje)...")
#epochs_combined = mne.concatenate_epochs([epochs_A, epochs_B])

print("Pre-processing zakończony sukcesem!")

Przetwarzanie Datasetu A (sesje 4, 8, 12)...
Using default location ~/mne_data for EEGBCI...
Creating /root/mne_data


Do you want to set the path:
    /root/mne_data
as the default EEGBCI dataset path in the mne-python config [y]/n? y
Attempting to create new mne-python configuration file:
/root/.mne/mne-python.json
Could not read the /root/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
Download complete in 36s (7.4 MB)
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Setting channel info structure...
Creating raw.info structure...
Readi

Download complete in 31s (7.3 MB)
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...


Download complete in 32s (7.4 MB)
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...


Download complete in 31s (7.3 MB)
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S004/S004R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S004/S004R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S004/S004R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...


...





Estimated consensus=0.60 and n_interpolate=32


  0%|          | Repairing epochs : 0/435 [00:00<?,       ?it/s]

Dropped 227 epochs: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 88, 96, 97, 104, 106, 107, 108, 109, 110, 114, 115, 133, 143, 144, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 264, 266, 268, 270, 271, 272, 274, 275, 276, 278, 282, 284, 286, 288, 291, 293, 295, 297, 299, 301, 307, 309, 313, 320, 322, 324, 326, 328,

/tmp/ipykernel_29297/3841918133.py:24: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_combined = mne.concatenate_epochs([epochs_A, epochs_B])


Applying baseline correction (mode: mean)
Pre-processing zakończony sukcesem!


In [3]:
import pandas as pd
import numpy as np

def extract_features_to_csv(epochs, output_filename):
    # ekstrakcja PSD dla pasma ruchowego (8-30 Hz)
    psd = epochs.compute_psd(method='welch', fmin=8.0, fmax=30.0, n_fft=256)
    psd_data = psd.get_data()

    # average i logarytmizacja
    features = np.log10(np.mean(psd_data, axis=2))
    df_features = pd.DataFrame(features, columns=epochs.ch_names)

    inv_event_id = {v: k for k, v in epochs.event_id.items()}

    # mapping tasks
    task_mapping = {
        'Rest': 'Task1',
        'Left_Fist': 'Task2',
        'Right_Fist': 'Task3',
        'Both_Fists': 'Task4',
        'Both_Feet': 'Task5'
    }

    df_features['Label'] = [task_mapping[inv_event_id[e]] for e in epochs.events[:, 2]]

    # saving
    df_features.to_csv(output_filename, index=False)
    print(f"Zapisano: {output_filename} | Kształt: {df_features.shape} (Próbki x Cechy+Etykieta)")
    return df_features

print("Rozpoczynam ekstrakcję cech do plików CSV...")

df_A = extract_features_to_csv(epochs_A, 'dataset_A_hands.csv')
df_B = extract_features_to_csv(epochs_B, 'dataset_B_feet_fists.csv')
df_C = extract_features_to_csv(epochs_combined, 'dataset_C_combined.csv')

print("Wszystkie zbiory danych zostały wyeksportowane")

Rozpoczynam ekstrakcję cech do plików CSV...
Effective window size : 1.600 (s)
Zapisano: dataset_A_hands.csv | Kształt: (387, 65) (Próbki x Cechy+Etykieta)
Effective window size : 1.600 (s)
Zapisano: dataset_B_feet_fists.csv | Kształt: (208, 65) (Próbki x Cechy+Etykieta)
Effective window size : 1.600 (s)
Zapisano: dataset_C_combined.csv | Kształt: (595, 65) (Próbki x Cechy+Etykieta)
Wszystkie zbiory danych zostały wyeksportowane
